## Notebook 3: AgentCore Memory on the Orchestrator

The pipeline you built in Notebook 1 was stateless, since every customer message started from scratch. Notebook 2 layered a durable session service on top so live-session events survive across processes. This notebook now attaches an **AgentCore Memory** resource to the same pipeline so it carries continuity across turns and across sessions, without changing a single specialist sub-agent.

Memory is applied selectively. Only the orchestrator sees memory, while the specialist sub-agents stay stateless and receive exactly the inputs defined by the orchestration contract. That keeps the specialists reusable by other orchestrators and keeps memory volume small (short snippets about the customer, not the full transcript of every sub-agent turn).

### Architecture

![Architecture with Memory](./images/architecture_with_memory_v2.png)

Three pieces coordinate memory injection and persistence, each with a clearly bounded role:

| Piece | Role | Lives in |
|---|---|---|
| `AgentCoreMemoryService` | Implements `BaseMemoryService`. `search_memory` combines short-term turns with long-term extractions; `add_session_to_memory` batches a session into one `create_event`. | `shared/memory.py` |
| `orchestrator_agent` | A flat `LlmAgent` that decides per turn which specialists to call. `preload_memory` is one of its tools, so retrieved context is injected as `<PAST_CONVERSATIONS>` before the model step on every turn. | `agents/orchestrator.py` |
| `after_agent_callback` | Fires once the orchestrator's turn ends. It calls `ctx.add_session_to_memory()`, which routes through the service and triggers async extraction. | `scenarios/scenario.py` |

Before each turn, `preload_memory` calls `search_memory` against AgentCore and the retrieved snippets arrive as a `<PAST_CONVERSATIONS>` block appended to the orchestrator's system prompt. The three specialists run unchanged from Notebook 1 and receive no memory context. Once the orchestrator's turn finishes, `after_agent_callback` batches the session's events into one `create_event` so short-term storage is immediate while `USER_PREFERENCE` and `SEMANTIC` extractions complete asynchronously about a minute later.

### Prerequisites

- Notebook 1 must have been run in this workspace, since the modules under `shared/`, `tools/`, `agents/`, and `scenarios/` are imported directly.
- AWS credentials for `us-west-2` with permissions for `bedrock-agentcore-control:*Memory*`, the Bedrock model permissions from Notebook 1, and `ssm:GetParameter` / `ssm:PutParameter` on `/agentcore-workshop/*`.
- Bedrock model access for Claude Haiku 4.5 and Claude Sonnet 4.6 in `us-west-2`.

This notebook creates one AgentCore resource (a Memory store named `CustomerSupportMemory`). The setup cell is idempotent; if a memory with that name already exists, the existing ID is reused. Every resource ID created in this and later notebooks is persisted to SSM Parameter Store under `/agentcore-workshop/`, so later notebooks pick them up without recreating anything.

### Step 0: Environment

All workshop dependencies were installed in Notebook 1. The imports below confirm the AgentCore Memory SDK is available before we build on top of it.

In [26]:
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

print("MemoryClient:", MemoryClient)
print("Strategies available:", [s.name for s in StrategyType])


MemoryClient: <class 'bedrock_agentcore.memory.client.MemoryClient'>
Strategies available: ['SEMANTIC', 'SUMMARY', 'USER_PREFERENCE', 'EPISODIC', 'CUSTOM']


### Step 1: The Raw AgentCore Memory API

`MemoryClient` is a thin SDK over the AgentCore Memory control plane and data plane. Four of its calls matter for this workshop:

| Call | Plane | Timing | Purpose |
|---|---|---|---|
| `create_memory_and_wait` | Control | One-time | Create a Memory resource with one or more extraction strategies. |
| `create_event` | Data | Synchronous | Append a conversation turn to the short-term buffer. Triggers async long-term extraction. |
| `get_last_k_turns` | Data | Synchronous | Read the last k turns verbatim (most-recent-first). |
| `retrieve_memories` | Data | Synchronous | Semantic search over already-extracted long-term memories in a namespace. |

AgentCore Memory separates synchronous short-term buffering from asynchronous long-term extraction. The workshop uses two extraction strategies, both running in the background after each `create_event`: `USER_PREFERENCE` extracts stable preferences such as "prefers the fastest option," and `SEMANTIC` extracts facts such as "order ORD-2026-0342 is an Acme XGzG 15." Each strategy owns a namespace template with `{actorId}` substitution, so memories for one customer never leak to another.

In [27]:
from shared.config import AWS_REGION

client = MemoryClient(region_name=AWS_REGION)  # note: region_name, not region
print("Connected to AgentCore Memory control plane in", AWS_REGION)


Connected to AgentCore Memory control plane in us-west-2


#### Step 1a: Extend `shared/config.py` with memory constants

Add the memory resource name and the strategy namespace templates so every downstream notebook sees the same definitions, plus `get_ssm_parameter` and `put_ssm_parameter` helpers that read and write resource IDs across the workshop. SSM Parameter Store is the workshop's single source of truth, and every notebook from here forward stores its resource IDs under the `/agentcore-workshop/` prefix.

In [28]:
%%writefile shared/config.py
# workshop/shared/config.py
from pathlib import Path

import boto3

AWS_REGION = "us-west-2"
DATA_DIR = Path(__file__).resolve().parent.parent / "data"

SSM_PREFIX = "/agentcore-workshop"

# Names introduced in earlier notebooks; carried forward here so a single
# shared/config.py module is the source of truth for every workshop resource.
SESSION_STORE_NAME = "CustomerSupportSessionStore"
MEMORY_NAME = "CustomerSupportMemory"
MEMORY_USER_PREFERENCE_NAMESPACE = "support/customer/{actorId}/preferences"
MEMORY_SEMANTIC_NAMESPACE = "support/customer/{actorId}/semantic"


def _ssm_client():
    return boto3.client("ssm", region_name=AWS_REGION)


def get_ssm_parameter(key: str) -> str:
    """Read a parameter from SSM under /agentcore-workshop/{key}.

    Raises ssm.exceptions.ParameterNotFound if the parameter does not exist.
    """
    response = _ssm_client().get_parameter(
        Name=f"{SSM_PREFIX}/{key}",
        WithDecryption=True,
    )
    return response["Parameter"]["Value"]


def put_ssm_parameter(key: str, value: str, encrypted: bool = False) -> None:
    """Write a parameter to SSM under /agentcore-workshop/{key}. Overwrites any existing value."""
    _ssm_client().put_parameter(
        Name=f"{SSM_PREFIX}/{key}",
        Value=value,
        Type="SecureString" if encrypted else "String",
        Overwrite=True,
    )


Overwriting shared/config.py


#### Step 1b: Idempotent memory creation

The cell below looks up an existing `CustomerSupportMemory` by name first; if none exists it creates one with the two strategies described above. `create_memory_and_wait` blocks (up to a minute) until the resource status flips to `ACTIVE` so subsequent cells can use it immediately.

In [29]:
import importlib
import shared.config as shared_config
importlib.reload(shared_config)
from shared.config import (
    MEMORY_NAME,
    MEMORY_USER_PREFERENCE_NAMESPACE,
    MEMORY_SEMANTIC_NAMESPACE,
    put_ssm_parameter,
)


def ensure_memory_resource() -> str:
    """Return the memory_id for CustomerSupportMemory, creating it if absent."""
    for existing in client.list_memories(max_results=100):
        existing_id = existing.get("id", "")
        if existing_id.startswith(MEMORY_NAME + "-"):
            return existing_id

    strategies = [
        {
            StrategyType.USER_PREFERENCE.value: {
                "name": "CustomerPreferences",
                "description": "Captures customer preferences and behavior",
                "namespaces": [MEMORY_USER_PREFERENCE_NAMESPACE],
            }
        },
        {
            StrategyType.SEMANTIC.value: {
                "name": "CustomerSupportSemantic",
                "description": "Stores facts from conversations",
                "namespaces": [MEMORY_SEMANTIC_NAMESPACE],
            }
        },
    ]

    memory = client.create_memory_and_wait(
        name=MEMORY_NAME,
        description="Workshop customer-support memory",
        strategies=strategies,
        event_expiry_days=90,
    )
    return memory["id"]


memory_id = ensure_memory_resource()
put_ssm_parameter("memory/memory_id", memory_id)
print("memory_id:", memory_id)
print("persisted to SSM at /agentcore-workshop/memory/memory_id")


memory_id: CustomerSupportMemory-ZQADL72SEQ
persisted to SSM at /agentcore-workshop/memory/memory_id


#### Step 1c: Inspect the strategies

`get_memory_strategies` returns the namespace templates. The `{actorId}` placeholder is substituted per-customer at retrieval time, which is how AgentCore enforces actor isolation on a single shared memory resource.

In [30]:
strategies = client.get_memory_strategies(memory_id)
for s in strategies:
    print(f"- {s['type']:<16} name={s['name']:<24} namespace={s['namespaces'][0]}")


- USER_PREFERENCE  name=CustomerPreferences      namespace=support/customer/{actorId}/preferences
- SEMANTIC         name=CustomerSupportSemantic  namespace=support/customer/{actorId}/semantic


#### Step 1d: Short-term recall via `create_event` and `get_last_k_turns`

Short-term memory is the verbatim turn buffer. Whatever was written via `create_event` is read back via `get_last_k_turns`, with no extraction and no LLM involvement. The cell below writes one synthetic turn for a demo actor and reads it back, exercising the same API the ADK wrapper calls internally.

In [31]:
DEMO_ACTOR = "C999-demo"
DEMO_SESSION = "demo-session-step1"

client.create_event(
    memory_id=memory_id,
    actor_id=DEMO_ACTOR,
    session_id=DEMO_SESSION,
    messages=[
        ("I want the fastest warranty option with no shipping delays.", "USER"),
        ("Understood. I will prioritise speed over cost.", "ASSISTANT"),
    ],
)

turns = client.get_last_k_turns(
    memory_id=memory_id, actor_id=DEMO_ACTOR, session_id=DEMO_SESSION, k=2
)
for i, turn in enumerate(turns):
    for msg in turn:
        print(f"[turn {i}] {msg['role']:<10} {msg['content']['text']}")


[turn 0] USER       I want the fastest warranty option with no shipping delays.
[turn 0] ASSISTANT  Understood. I will prioritise speed over cost.
[turn 1] USER       I want the fastest warranty option with no shipping delays.
[turn 1] ASSISTANT  Understood. I will prioritise speed over cost.


#### Step 1e: Long-term recall via `retrieve_memories`

Long-term extraction runs asynchronously and takes about a minute per `create_event`, which is why the acceptance test later in the notebook includes an explicit wait. `retrieve_memories` is a semantic search over whatever has already been extracted into the supplied namespace, so immediately after the very first `create_event` the result list is often empty because extraction has not yet finished.

In [32]:
namespace = MEMORY_USER_PREFERENCE_NAMESPACE.replace("{actorId}", DEMO_ACTOR)
results = client.retrieve_memories(
    memory_id=memory_id, namespace=namespace, query="speed preference", top_k=3
)
if not results:
    print("No extracted preferences yet -- long-term extraction runs ~1 minute after create_event.")
else:
    for i, mem in enumerate(results):
        print(f"[{i}]", mem.get("content", {}).get("text", ""))


[0] {"context":"The user explicitly stated that they want the fastest warranty option with no shipping delays, prioritizing speed over cost.","preference":"Prefers fastest shipping and warranty options with no delays, prioritizes speed over cost","categories":["shopping","shipping","warranty","service speed"]}


### Step 2: Implement the Memory Service

ADK exposes memory through `BaseMemoryService`, which has two abstract methods:

```python
async def search_memory(self, *, app_name, user_id, query) -> SearchMemoryResponse
async def add_session_to_memory(self, session) -> None
```

Implementing these against the four `MemoryClient` primitives from Step 1 gives any ADK `Runner` a working memory backend:

| `BaseMemoryService` method | Delegates to |
|---|---|
| `search_memory` | `get_last_k_turns` (short-term) plus `retrieve_memories` per strategy namespace (long-term) |
| `add_session_to_memory` | `create_event` with all (text, role) pairs from the session's events |

`create_event` accepts many `(text, role)` tuples in one call and ADK fires `add_session_to_memory` once at the end of a pipeline run rather than per sub-agent, so batching the whole session into a single event keeps traffic to AgentCore low and lines up the async extraction boundary with a clean pipeline completion. The service is written to `shared/memory.py` so subsequent notebooks can import it unchanged.

In [33]:
%%writefile shared/memory.py
# workshop/shared/memory.py
from google.adk.memory.base_memory_service import (
    BaseMemoryService,
    SearchMemoryResponse,
)
from google.adk.memory.memory_entry import MemoryEntry
from google.genai import types

from shared.config import AWS_REGION


class AgentCoreMemoryService(BaseMemoryService):
    """Bridges ADK's memory interface to an AgentCore Memory resource.

    Short-term recall comes from ``get_last_k_turns`` (the verbatim turn buffer).
    Long-term recall comes from ``retrieve_memories`` across every strategy
    namespace registered on the resource.
    """

    def __init__(
        self,
        *,
        memory_id: str,
        region: str = AWS_REGION,
        short_term_turns: int = 5,
        long_term_top_k: int = 5,
    ) -> None:
        self._memory_id = memory_id
        self._region = region
        self._short_term_turns = short_term_turns
        self._long_term_top_k = long_term_top_k
        self._client = None
        self._namespace_cache: dict[str, str] | None = None

    @property
    def client(self):
        if self._client is None:
            from bedrock_agentcore.memory import MemoryClient

            self._client = MemoryClient(region_name=self._region)
        return self._client

    def _namespaces(self) -> dict[str, str]:
        if self._namespace_cache is None:
            strategies = self.client.get_memory_strategies(self._memory_id)
            self._namespace_cache = {
                s["type"]: s["namespaces"][0] for s in strategies
            }
        return self._namespace_cache

    async def search_memory(
        self, *, app_name: str, user_id: str, query: str
    ) -> SearchMemoryResponse:
        memories: list[MemoryEntry] = []

        turns = self.client.get_last_k_turns(
            memory_id=self._memory_id,
            actor_id=user_id,
            session_id=app_name,
            k=self._short_term_turns,
        )
        for turn in turns or []:
            for msg in turn:
                text = msg.get("content", {}).get("text", "")
                if not text:
                    continue
                role = msg.get("role", "user").lower()
                memories.append(
                    MemoryEntry(
                        content=types.Content(
                            parts=[types.Part(text=text)], role=role
                        ),
                        author=msg.get("role", "unknown"),
                    )
                )

        for strategy_type, template in self._namespaces().items():
            namespace = template.replace("{actorId}", user_id)
            results = self.client.retrieve_memories(
                memory_id=self._memory_id,
                namespace=namespace,
                query=query,
                top_k=self._long_term_top_k,
            )
            for mem in results or []:
                content = mem.get("content", {}) if isinstance(mem, dict) else {}
                text = (
                    content.get("text", "").strip()
                    if isinstance(content, dict)
                    else str(content).strip()
                )
                if not text:
                    continue
                memories.append(
                    MemoryEntry(
                        content=types.Content(parts=[types.Part(text=text)]),
                        author=strategy_type,
                    )
                )

        return SearchMemoryResponse(memories=memories)

    async def add_session_to_memory(self, session) -> None:
        messages: list[tuple[str, str]] = []
        for event in session.events:
            if not event.content or not event.content.parts:
                continue
            texts = [p.text for p in event.content.parts if p.text]
            if not texts:
                continue
            role = "USER" if event.author == "user" else "ASSISTANT"
            messages.append((" ".join(texts), role))

        if not messages:
            return

        self.client.create_event(
            memory_id=self._memory_id,
            actor_id=session.user_id,
            session_id=session.id,
            messages=messages,
        )


Overwriting shared/memory.py


### Step 3: Insert the Memory-Aware Orchestrator Agent

Notebook 1's orchestrator was a flat `LlmAgent` with three sibling `AgentTool`s (customer_order, diagnosis, cost_recommendation). Adding memory does not change that shape. We add `preload_memory` as a fourth tool on the same orchestrator. There is no separate ack agent and no `SequentialAgent` wrapper, since the same orchestrator that decides whether to call diagnosis decides whether to call `preload_memory`.

`preload_memory` is a pre-LLM hook rather than a regular tool call. When the orchestrator "uses" it, ADK runs `search_memory` against AgentCore Memory before the model step and injects any matching snippets as a `<PAST_CONVERSATIONS>` block in the next prompt, so the orchestrator never sees a `function_response` for it. That's why we keep the call cheap and keep the prompt's grounding rules strict about what the model is allowed to quote from that block.

The orchestrator module still exposes `root_agent` so Notebook 1's public surface is unchanged. The notebook wires the memory save callback later by assigning `root_agent.after_agent_callback = persist_session_callback`.

In [34]:
%%writefile agents/orchestrator.py
# workshop/agents/orchestrator.py
from google.adk.agents import Agent
from google.adk.tools import preload_memory
from google.adk.tools.agent_tool import AgentTool

from shared.models import SONNET

from agents.cost_recommendation_agent import cost_recommendation_agent
from agents.customer_order_agent import customer_order_agent
from agents.diagnosis_agent import diagnosis_agent


ORCHESTRATOR_PROMPT = """You are a customer-support assistant. You speak directly
to the customer. Be concise, accurate, and professional.

Tools:
- customer_order_agent: verify a customer/order pair. Returns the verified SKU.
- diagnosis_agent: identifies the defect, defect_id, and whether a software fix
  exists. Pass the verified SKU and a one-sentence symptom.
- cost_recommendation_agent: returns ranked-options JSON. Pass the verified SKU
  and the diagnosis output.
- preload_memory: searches prior sessions for this customer. Call when the customer references "before" / "earlier" / "last time" without giving the order id this session.

Initial-issue flow (strict order, one tool call per assistant turn):
1. Call customer_order_agent. Wait for its reply; read the verified SKU.
2. Call diagnosis_agent with that SKU and the symptom. Wait for its reply.
3. Call cost_recommendation_agent with the SKU and the diagnosis. Wait for its reply.
4. Compose the customer reply.

Do not issue calls for steps 1 and 2 in the same assistant turn -- diagnosis_agent
needs the SKU returned by step 1.

For follow-ups about something already analyzed this session, answer from history --
do not call any tool. If the customer raises a new symptom or defect, re-run
diagnosis_agent + cost_recommendation_agent; do not reuse prior options or prices.

Grounding:
- Use only facts from this turn's tool outputs or this session's history.
- Quote product names, order ids, prices, and defect ids verbatim. Never
  substitute brand names.
- If diagnosis returns a defect_id, include it. If no known defect was found,
  say so honestly -- do not invent a hardware failure.
- If a fact is missing, say so and ask the customer.
- If <PAST_CONVERSATIONS> shows the customer asked a question but did not decide on an option, do not say a decision was made.
- Do not introduce prior facts (preferences, additional issues, prices) that are not present verbatim in <PAST_CONVERSATIONS> or earlier turns of this session.

Format: plain conversational prose, no emoji, no markdown headings, tables, or horizontal rules.
Bold only on the recommended option's name when listing 2+ options. Bullets only
when listing 2+ customer-facing options. Initial reply under 180 words; follow-ups
under 80 words. Never expose tool JSON."""


orchestrator_agent = Agent(
    name="orchestrator",
    model=SONNET,
    description="Customer-support assistant. Calls specialist tools when needed and composes the customer reply.",
    instruction=ORCHESTRATOR_PROMPT,
    tools=[
        AgentTool(agent=customer_order_agent),
        AgentTool(agent=diagnosis_agent),
        AgentTool(agent=cost_recommendation_agent),
        preload_memory,
    ],
)

root_agent = orchestrator_agent


Overwriting agents/orchestrator.py


### Step 4: Extend the Scenario Helpers

Notebook 1's `scenarios/scenario.py` exposed a single `run_scenario(root_agent)` (one-shot, in-memory session, no memory service). Demonstrating memory continuity needs a reusable `Runner` that keeps a session across turns plus a way to attach a `memory_service`, so we extend the module with four small helpers. We also swap the `Runner`'s session service from `InMemorySessionService()` to `get_session_service()`, which Notebook 2 wrote to `shared/session.py`. The `Runner` now reads and writes every `Event` through AgentCore Memory's session log, so a kernel restart no longer resets the conversation.

| Helper | What it does |
|---|---|
| `persist_session_callback(callback_context)` | Coroutine for `after_agent_callback` that writes the finished session into the memory service. |
| `make_runner(root_agent, memory_service=None)` | Constructs a fresh `Runner` with `AgentCoreSessionService` (from `shared/session.py`), optionally wiring in a memory service. |
| `run_turn(runner, user_id, session_id, message)` | Dispatches a single user message and returns the final assistant text, so we can drive multiple turns in one session. |
| `run_scenario(root_agent, memory_service=None, user_id=..., prompt=CUSTOMER_MESSAGE)` | Preserves Notebook 1's signature plus optional memory wiring. The `prompt` kwarg defaults to the standard customer message but lets later notebooks (Notebook 4 onwards) reuse the same helper for ad-hoc prompts without duplicating `Runner` setup. |

In [35]:
%%writefile scenarios/scenario.py
# workshop/scenarios/scenario.py
from google.adk.agents.callback_context import CallbackContext
from google.adk.runners import Runner
from google.genai import types

from shared.session import get_session_service

CUSTOMER_MESSAGE = (
    "Hi, my order ORD-2026-0342 (customer ID C001) has a problem. "
    "My Acme XGzG 15 screen keeps flickering. I already restarted it twice. "
    "What are my options and how much will they cost?"
)

EXPECTED_CUSTOMER_ID = "C001"
EXPECTED_ORDER_ID = "ORD-2026-0342"

APP_NAME = "customer_support_workshop"
USER_ID = "workshop_user"


async def persist_session_callback(callback_context: CallbackContext) -> None:
    """Fires after the pipeline ends; writes the session into the memory service."""
    await callback_context.add_session_to_memory()


def make_runner(root_agent, memory_service=None) -> Runner:
    return Runner(
        app_name=APP_NAME,
        agent=root_agent,
        session_service=get_session_service(),
        memory_service=memory_service,
    )


async def run_turn(
    runner: Runner, *, user_id: str, session_id: str, message: str
) -> str:
    content = types.Content(
        role="user", parts=[types.Part.from_text(text=message)]
    )
    final_text = ""
    async for event in runner.run_async(
        user_id=user_id, session_id=session_id, new_message=content
    ):
        if event.is_final_response() and event.content and event.content.parts:
            for part in event.content.parts:
                if part.text:
                    final_text = part.text
    return final_text


async def run_scenario(
    root_agent,
    *,
    memory_service=None,
    user_id: str = USER_ID,
    prompt: str = CUSTOMER_MESSAGE,
) -> str:
    runner = make_runner(root_agent, memory_service=memory_service)
    session = await runner.session_service.create_session(
        app_name=APP_NAME, user_id=user_id
    )
    return await run_turn(
        runner,
        user_id=user_id,
        session_id=session.id,
        message=prompt,
    )


Overwriting scenarios/scenario.py


### Step 5: First Check, Happy-Path Scenario Still Passes

Memory wiring should not change the system's existing behavior. We attach the memory service, wire `persist_session_callback` onto the pipeline, and re-run the same customer message from Notebook 1. The downstream specialists produce the same structured outputs and the same ranked recommendation; the only observable difference is an extra short acknowledgement at the top of the output emitted by `orchestrator_agent`.

In [36]:
from shared.memory import AgentCoreMemoryService
from agents.orchestrator import root_agent
from scenarios.scenario import (
    persist_session_callback,
    run_scenario,
)

memory_service = AgentCoreMemoryService(memory_id=memory_id)
root_agent.after_agent_callback = persist_session_callback

final_text = await run_scenario(
    root_agent,
    memory_service=memory_service,
    user_id="C001",
)
print(final_text)


Great news — your issue has a straightforward and free fix. Here is what I found:

Your Acme XGzG 15 NK30 is affected by known defect DEF-2024-1122, a firmware bug that causes screen flickering at refresh rates above 60Hz due to a GPU-display sync conflict. Restarting does not resolve it because it is tied to the BIOS, not a transient glitch.

Your option is a **self-service BIOS update to version 1.9.2 or later** using the Acme Support utility. It costs $0.00 and can be done today — no shipping or wait time involved.

If the flickering persists after the update, there is a secondary possibility (defect DEF-2025-0418 — backlight bleed requiring a panel replacement), but that should only be explored if the firmware fix does not resolve the issue. I do not have confirmed pricing for that repair at this time.

Would you like guidance on how to run the BIOS update?


### Step 6: Second Check, Short-Term Recall Within a Session

Continuity within a session means a second message in the same session should recall details from the first without the customer re-stating them. Short-term recall is synchronous, so `create_event` writes turn 1 to AgentCore Memory before turn 2 starts and `preload_memory` on turn 2 already sees the buffered turn. We dispatch two related messages in one session and confirm the second response is continuity-aware.

In [37]:
from scenarios.scenario import APP_NAME, make_runner, run_turn

runner = make_runner(root_agent, memory_service=memory_service)
short_term_session = await runner.session_service.create_session(
    app_name=APP_NAME, user_id="C001"
)

turn_1 = await run_turn(
    runner,
    user_id="C001",
    session_id=short_term_session.id,
    message=(
        "Hi, my order ORD-2026-0342 (customer ID C001) has a problem. "
        "My Acme XGzG 15 screen keeps flickering. What are my options?"
    ),
)
print("=== Turn 1 final response ===")
print(turn_1[:1200])


=== Turn 1 final response ===
Great news — there is a straightforward, no-cost fix for your Acme XGzG 15 NK30.

Your flickering is a known firmware issue (defect DEF-2024-1122), caused by a GPU-display sync conflict at refresh rates above 60Hz. The fix is a self-service BIOS update to version 1.9.2 or later using the Acme Support utility — it costs nothing and can be done immediately today.

If the flickering continues after the update, there is a secondary known defect (DEF-2025-0418) involving a hardware panel issue, which would require a service center visit. We can cross that bridge if needed.

Would you like guidance on how to run the BIOS update?


In [38]:
turn_2 = await run_turn(
    runner,
    user_id="C001",
    session_id=short_term_session.id,
    message="Actually, I only care about the fastest option. Which one should I pick?",
)
print("=== Turn 2 final response ===")
print(turn_2[:1200])

continuity_markers = ["Acme XGzG", "ORD-2026-0342", "flicker", "firmware"]
matched = [m for m in continuity_markers if m.lower() in turn_2.lower()]
print()
print(f"Continuity markers present in turn 2: {matched}")
assert matched, "Turn 2 did not reference any turn-1 context; short-term recall failed."


18:00:57 - LiteLLM:WARNING: factory.py:4714 - Potential consecutive user/tool blocks. Trying to merge. If error occurs, please set a 'assistant_continue_message' or set 'modify_params=True' to insert a dummy assistant message for bedrock calls.
=== Turn 2 final response ===
Based on your previous sessions and the analysis just completed, there is only one option available for your Acme XGzG 15 NK30, and it also happens to be the fastest: a self-service BIOS update to version 1.9.2 or later using the Acme Support utility. It addresses defect DEF-2024-1122, costs nothing, and can be done right now with no wait time.

Would you like step-by-step instructions to run the update?

Continuity markers present in turn 2: ['Acme XGzG']


### Step 7: Third Check, Cross-Session Long-Term Recall

Cross-session continuity is what makes memory feel useful over time, since a customer who messages again tomorrow should surface extracted preferences and facts from previous conversations even in a brand-new session. This is the asynchronous side of the memory system because the `USER_PREFERENCE` and `SEMANTIC` strategies run in the background after each `create_event` and take around a minute to produce extracted records.

The acceptance test runs in four steps:

1. Seed a preference-shaped message for a dedicated test actor (`C001-long-term-test`), giving the strategy engine a stated preference for speed over cost.
2. Wait long enough for async extraction (the cell uses 90 seconds; the service documents about a minute).
3. Inspect the extracted memories via the raw `retrieve_memories` API to prove extraction succeeded independently of any agent.
4. Start a brand-new session for the same actor and issue a follow-up that benefits from the extracted preference.

On the recall turn, the orchestrator typically re-verifies the order via `customer_order_agent` (the customer ID and order ID arrive again) and weaves the recalled preference verbatim into its reply, without inventing a prior decision the customer never made.

Long-term extraction is a background AWS process and can occasionally take longer than 90 seconds. If `retrieve_memories` returns an empty list, re-running the extraction-check cell after another minute usually surfaces results.

In [39]:
LONG_TERM_ACTOR = "C001-long-term-test"

seed_runner = make_runner(root_agent, memory_service=memory_service)
seed_session = await seed_runner.session_service.create_session(
    app_name=APP_NAME, user_id=LONG_TERM_ACTOR
)
seed_response = await run_turn(
    seed_runner,
    user_id=LONG_TERM_ACTOR,
    session_id=seed_session.id,
    message=(
        "Hi, my order ORD-2026-0342 (customer ID C001) has a screen flicker issue. "
        "One preference: I always want the fastest resolution, even if another option is cheaper."
    ),
)
print("Seed session final response:")
print(seed_response[:600])


Seed session final response:
Here is what we found for your Acme XGzG 15 NK30 (order ORD-2026-0342).

Your screen flicker is caused by a known firmware bug, defect ID DEF-2024-1122, which creates a GPU-display sync conflict at refresh rates above 60Hz. The great news is a fix is available right now at no cost: **Self-service BIOS update to version 1.9.2 or later** using the Acme Support utility. This is also the fastest option, resolving the issue immediately with zero downtime.

The diagnosis also flagged a secondary known issue, defect ID DEF-2025-0418, involving hardware backlight bleed along the bottom edge. This is a


In [40]:
import time

WAIT_SECONDS = 90
print(f"Waiting {WAIT_SECONDS}s for USER_PREFERENCE / SEMANTIC extraction to complete...")
time.sleep(WAIT_SECONDS)

pref_namespace = MEMORY_USER_PREFERENCE_NAMESPACE.replace("{actorId}", LONG_TERM_ACTOR)
semantic_namespace = MEMORY_SEMANTIC_NAMESPACE.replace("{actorId}", LONG_TERM_ACTOR)

pref_hits = client.retrieve_memories(
    memory_id=memory_id, namespace=pref_namespace,
    query="preference on speed versus cost", top_k=5,
)
semantic_hits = client.retrieve_memories(
    memory_id=memory_id, namespace=semantic_namespace,
    query="Acme XGzG 15 screen flicker", top_k=5,
)

print(f"USER_PREFERENCE hits: {len(pref_hits)}")
for m in pref_hits:
    print(" -", m.get("content", {}).get("text", ""))
print(f"SEMANTIC hits: {len(semantic_hits)}")
for m in semantic_hits:
    print(" -", m.get("content", {}).get("text", ""))


Waiting 90s for USER_PREFERENCE / SEMANTIC extraction to complete...
USER_PREFERENCE hits: 1
 - {"context":"The user explicitly stated a clear preference regarding resolution approach, emphasizing speed over cost considerations.","preference":"Always wants the fastest resolution, even if another option is cheaper","categories":["customer service","problem resolution","decision-making"]}
SEMANTIC hits: 4
 - The user owns an Acme XGzG 15 NK30 device that has a screen flicker issue caused by firmware bug DEF-2024-1122.
 - The user's order number is ORD-2026-0342.
 - The user's customer ID is C001.
 - The user always prefers the fastest resolution, even if another option is cheaper.


In [41]:
# New session, same actor -- verifies that long-term memory surfaces in a fresh Runner lifecycle.
followup_runner = make_runner(root_agent, memory_service=memory_service)
followup_session = await followup_runner.session_service.create_session(
    app_name=APP_NAME, user_id=LONG_TERM_ACTOR
)
followup_response = await run_turn(
    followup_runner,
    user_id=LONG_TERM_ACTOR,
    session_id=followup_session.id,
    message=(
        "Hi again -- same customer ID C001, same order ORD-2026-0342. "
        "Can you confirm which resolution path best matches how I like things handled?"
    ),
)
print("Follow-up (new session) final response:")
print(followup_response)


Follow-up (new session) final response:
No need to re-run diagnostics or pricing — everything from your last session is already on file.

Your account is verified: order ORD-2026-0342, Acme XGzG 15 NK30, delivered 63 days ago.

Based on your stated preference — fastest resolution, even if it costs more — the best match for you would be whichever option from your prior session offered the quickest turnaround, not the cheapest one. Your screen flicker issue was tied to firmware bug DEF-2024-1122, and if a software fix was available, that would typically be the fastest path.

That said, I don't have the specific ranked options and prices from your last session in front of me right now. Would you like me to re-run the full recommendation so I can give you a precise, current answer with the fastest option clearly identified?


### Step 8: Fourth Check, Actor Isolation

The namespace templates end with `/{actorId}/...`, so two customers sharing the same memory resource should never see each other's history even on the same memory resource ID. The cell below verifies this by querying the `USER_PREFERENCE` namespace with a different `actor_id` and asserting that zero memories are returned.

In [42]:
OTHER_ACTOR = "C999-isolation-test"
other_namespace = MEMORY_USER_PREFERENCE_NAMESPACE.replace("{actorId}", OTHER_ACTOR)

other_hits = client.retrieve_memories(
    memory_id=memory_id, namespace=other_namespace,
    query="preference on speed versus cost", top_k=5,
)
print(f"USER_PREFERENCE hits for {OTHER_ACTOR}: {len(other_hits)}")
for m in other_hits:
    print(" -", m.get("content", {}).get("text", ""))

assert len(other_hits) == 0, (
    f"Actor isolation failed: {OTHER_ACTOR} namespace returned {len(other_hits)} memories."
)
print()
print("Actor isolation confirmed: a different actor_id sees zero prior memories.")


USER_PREFERENCE hits for C999-isolation-test: 0

Actor isolation confirmed: a different actor_id sees zero prior memories.


### What You Built

- `CustomerSupportMemory`, one AgentCore Memory resource per account, created idempotently with `USER_PREFERENCE` and `SEMANTIC` strategies. The `memory_id` is persisted to SSM at `/agentcore-workshop/memory/memory_id`.
- `shared/memory.py`, an `AgentCoreMemoryService(BaseMemoryService)` that combines short-term and long-term recall in `search_memory` and batches finished sessions into one `create_event` in `add_session_to_memory`.
- A fourth tool on the orchestrator: `preload_memory`, sitting alongside the three sibling `AgentTool`s. It auto-injects `<PAST_CONVERSATIONS>` context before every model call and does not show up as a `function_response`.
- An `after_agent_callback` hook that wires the memory service into the pipeline lifecycle so finished sessions trigger async extraction automatically.
- Four acceptance checks passing against real AgentCore Memory: happy path, two-turn short-term recall, cross-session long-term recall, and actor isolation.

`preload_memory` runs before the LLM rather than as a tool call, so the model never decides whether to search memory and never gets that decision wrong. `BaseMemoryService` is the ADK extension point, where implementing two methods plugs any memory backend into any `Runner` without touching agent code.

### Cleanup

`CustomerSupportMemory` is the one persistent AWS resource this notebook creates. AgentCore Memory has no idle compute charge and persists indefinitely, but if you want a clean account, delete the resource from the AgentCore console (or call `bedrock-agentcore-control:DeleteMemory` with the `memory_id` from `/agentcore-workshop/memory/memory_id`) and clear that SSM key. `bash reset_env.sh --full` removes the SSM pointer but does not delete the underlying Memory resource.

### Next Up

**[Notebook 4: AgentCore Code Interpreter](04_agentcore_code_interpreter.ipynb)** introduces the next AgentCore service. The cost & recommendation agent stops ranking options in-prompt and instead generates a Python script that runs inside the AgentCore Code Interpreter sandbox, making ranking logic testable and auditable without touching any of the memory wiring introduced here.